# 01 – Data Acquisition: Hourly Temperature Observations (2012)

## Objective

This notebook documents the acquisition of raw hourly temperature observations
for Oklahoma (starting with Oklahoma City) for the year 2012.

Raw observational data is collected from multiple sources including:

- NOAA surface observation stations
- Oklahoma Mesonet stations
- Potential additional observational networks

The **data retrieval itself is performed by reusable Python scripts**
located in the `scripts/` directory. This notebook demonstrates how
those scripts are executed and verifies the results.

## Goals of this Stage

The purpose of this stage is **not analysis**.

Instead we focus on building a **reproducible data foundation** for the
Weather Agreement Lab.

Specifically we will:

- Retrieve reproducible raw data
- Preserve the original files exactly as received
- Store the files in `data/raw/`
- Document station metadata and acquisition details

## Role in the Weather Agreement Lab Pipeline

This notebook establishes the **data acquisition layer** of the project.

Downstream stages will:

- Clean and align the observational datasets
- Measure agreement between agencies
- Explore forecast uncertainty
- Develop machine-learning based fusion models


# Suggested Updated Version

```
# Environment Setup (Run Before Executing This Notebook)

This notebook is part of the **Weather Agreement Lab**, a reproducible
data science workflow for studying agreement between weather observation
networks.

The project follows a **research-style pipeline**:

```

```
scripts   → retrieve and process data
notebooks → explore and explain results
report    → produce the final LaTeX write-up
```

```

Before running this notebook, create a Python virtual environment and
install the required dependencies.

---

## 1. Clone the Repository

```

```bash
git clone git@github.com:jeremy-evert/handson-ml3.git
cd handson-ml3/Weather_Agreement_Lab
```

---

## 2. Create a Virtual Environment

Linux / macOS

```bash
python3 -m venv .venv
```

Windows

For the first time, run this command.

```powershell
Set-ExecutionPolicy -ExecutionPolicy Bypass -Scope Process
```

```powershell
python -m venv .venv
```

---

## 3. Activate the Virtual Environment

Linux / macOS

```bash
source .venv/bin/activate
```

Windows

```powershell
.venv\Scripts\activate
```

---

## 4. Install Required Packages

````

```bash
pip install --upgrade pip
pip install -r requirements_weather_lab.txt
````

---

## 5. Register the Jupyter Kernel

This allows the notebook to run using the virtual environment.

```bash
python -m ipykernel install --user \
  --name weather_lab_env \
  --display-name "Python (Weather Lab)"
```

---

## 6. Open the Project

Recommended workflow using **VS Code**:

```bash
code .
```

Then:

1. Open `01_data_pull.ipynb`
2. Click **Select Kernel** (upper right)
3. Choose **Python (Weather Lab)**

---

## Alternative: Launch Jupyter Directly

```bash
jupyter notebook
```

Open:

```
01_data_pull.ipynb
```

---

## Key Dependencies

Defined in:

```
requirements_weather_lab.txt
```

Major packages used in this lab include:

* pandas
* numpy
* matplotlib
* scikit-learn
* requests
* jupyter / ipykernel

---

## Project Layout

```
Weather_Agreement_Lab
│
├── notebooks
│   ├── 01_data_pull.ipynb
│   ├── 02_alignment_and_cleaning.ipynb
│   ├── 03_forecast_uncertainty.ipynb
│   ├── 04_agreement_metrics.ipynb
│   └── 05_data_fusion.ipynb
│
├── scripts
│   ├── pull_noaa.py
│   ├── clean_data.py
│   ├── compute_metrics.py
│   └── fusion_model.py
│
├── data
│   ├── raw
│   ├── processed
│   └── analysis
│
└── requirements_weather_lab.txt
```

Notebook progression:

1️⃣ Data Acquisition
2️⃣ Cleaning & Alignment
3️⃣ Forecast Uncertainty
4️⃣ Agreement Metrics
5️⃣ Data Fusion

In [ ]:
from scripts.session_diagnostics import run_session_diagnostics

run_session_diagnostics()

In [ ]:
from scripts.config import DATA_RAW, DATA_PROCESSED
from scripts.config import ensure_directories
ensure_directories()

print("Raw data directory:")
print(DATA_RAW)

## NOAA API Token Setup

This lab retrieves weather observations using the **NOAA Climate Data API**.

Access to this API requires a free NOAA API token.

---

### Step 1 — Request a Token

Visit the NOAA token request page:

https://www.ncdc.noaa.gov/cdo-web/token

Complete the short form and NOAA will email you an API token.

---

### Step 2 — Create a `.env` File

Inside the **Weather_Agreement_Lab** directory create a file named:

```

```
.env
```

Example project structure:

```
```

Weather_Agreement_Lab
│
├── .env
├── notebooks/
│   └── 01_data_pull.ipynb
├── scripts/
├── requirements_weather_lab.txt

```

---

### Step 3 — Add Your Token

Add the following line to the `.env` file:

```

```
NOAA_API_TOKEN=your_token_goes_here
```

Example:

```
```

NOAA_API_TOKEN=ABCD1234EFGH5678

```

---

### Step 4 — Keep the Token Private

The `.env` file must **never be committed to Git**.

Ensure `.gitignore` contains:

```

```
.env
```

---

### Step 5 — Run the Notebook

The data acquisition scripts automatically load the API token
from the `.env` file.

If the token is missing or invalid, the scripts will stop with a
clear error message.

In [ ]:
from scripts.noaa_auth import get_noaa_token

print("Token prefix:", get_noaa_token()[:4] + "...")

In [ ]:
from scripts.config import show_data_status

show_data_status()

In [ ]:
from scripts.noaa_stations import find_station_by_city

find_station_by_city("Oklahoma")

In [ ]:
import pandas as pd

inventory_url = "https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv"

stations = pd.read_csv(inventory_url)

ok = stations[
    stations["STATION NAME"].str.contains("OKLAHOMA CITY", case=False, na=False)
]

ok[["USAF","WBAN","STATION NAME","BEGIN","END"]]

In [ ]:
from scripts.stations import load_okc_2012

df = load_okc_2012()
df.head()

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Column names:")
print(df.columns)

In [ ]:
print("First timestamp:", df["DATE"].min())
print("Last timestamp:", df["DATE"].max())

In [ ]:
df[["DATE","TMP","WND","VIS"]].head()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(10)